# Entendimiento de imágenes con LLMs multimodales

**Lección 2 · Clase 5.3** — cómo pasarle imágenes a GPT-5 (u otro modelo multimodal) y conversar sobre lo que "ve".

La estrategia es simple: la imagen entra como **un bloque más del mensaje**, en una de dos formas:

1. **URL** — si la imagen ya vive en internet.
2. **Base64** — el archivo local codificado como string.

Usamos la interfaz estándar de LangChain: los bloques de imagen viajan dentro de un `HumanMessage`, junto al texto.

In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta:
# %pip install -q langchain-openai==1.3.5 langchain-core==1.4.9 python-dotenv==1.2.2 pillow==12.3.0
from dotenv import load_dotenv
import os

# Carga OPENAI_API_KEY desde .env si existe (local); en Colab usa Secrets.
load_dotenv()

try:
    from google.colab import userdata  # type: ignore
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
except Exception:
    pass

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY presente:", True)
else:
    print("⚠️ Falta OPENAI_API_KEY — usa un archivo .env o los Secrets de Colab.")


## La imagen de prueba

Un clásico de la TV chilena — si el archivo local no está (p. ej. en Colab), se descarga desde Wikimedia.

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

image_path = Path("Cachureos2020.jpg")
if not image_path.exists():
    import urllib.request
    urllib.request.urlretrieve(
        "https://upload.wikimedia.org/wikipedia/commons/8/8c/Cachureos2020.jpg",
        image_path,
    )

display(Image.open(image_path).reduce(2))

## Opción 1 — base64 dentro de un `HumanMessage`

El contenido de un mensaje puede ser una **lista de bloques**: texto e imágenes conviven en el mismo mensaje. Para un archivo local, se codifica en base64 y se declara su `mime_type`.

In [ ]:
import base64
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5")

image_data = base64.b64encode(image_path.read_bytes()).decode("utf-8")

message = HumanMessage(
    content=[
        {"type": "text", "text": "Describe brevemente la imagen que estás viendo. ¿De qué show es? (hint: es chileno)"},
        {
            "type": "image",
            "source_type": "base64",
            "data": image_data,
            "mime_type": "image/jpeg",
        },
    ]
)
respuesta = llm.invoke([message])
print(respuesta.content)

## Opción 2 — URL directa

Si la imagen ya está en internet no hay que codificar nada: el bloque lleva la URL y el proveedor la descarga por su cuenta.

In [ ]:
message_url = HumanMessage(
    content=[
        {"type": "text", "text": "¿Qué personajes u objetos destacan en esta imagen? Enumera 3."},
        {
            "type": "image",
            "source_type": "url",
            "url": "https://upload.wikimedia.org/wikipedia/commons/8/8c/Cachureos2020.jpg",
        },
    ]
)
print(llm.invoke([message_url]).content)

## Cierre

El mismo patrón sirve para cualquier modelo multimodal (Claude, Gemini, etc.): LangChain normaliza los bloques de imagen entre proveedores. En la **lección 3** damos vuelta la flecha: en vez de entender imágenes, las generamos. Y en la **lección 4** usamos esta misma capacidad para un caso de negocio real: extraer datos estructurados de un formulario escaneado.